# 00 — ROCm sanity check

**Goal:** verify the ROCm + PyTorch + Transformers stack is healthy on Strix Halo before doing any real work.

**Pass criteria:**
1. `torch.cuda.is_available()` returns `True` (ROCm exposes the CUDA API).
2. A small model (Pythia-70m) loads onto the device.
3. A forward pass on an Arabic prompt produces logits without error.

If any cell errors, fix the environment before moving on. This notebook is the contract for what 'environment works' means in this repo.

## 1. Torch + device check

In [ ]:
import torch

print(f"torch version:        {torch.__version__}")
print(f"CUDA-API available:   {torch.cuda.is_available()}")  # True under ROCm
print(f"Device count:         {torch.cuda.device_count() if torch.cuda.is_available() else 0}")

if torch.cuda.is_available():
    print(f"Device name:          {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / (1024**3)
    print(f"Total memory:         {total_gb:.1f} GB")

# ROCm-specific sanity
if hasattr(torch.version, "hip") and torch.version.hip is not None:
    print(f"HIP version:          {torch.version.hip}")
elif hasattr(torch.version, "cuda") and torch.version.cuda is not None:
    print(f"CUDA version:         {torch.version.cuda}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device:         {device}")

## 2. Tiny tensor op (catches obvious driver/kernel issues)

In [ ]:
x = torch.randn(1024, 1024, device=device)
y = torch.randn(1024, 1024, device=device)
z = x @ y
print(f"matmul OK — output shape {tuple(z.shape)}, device {z.device}, dtype {z.dtype}")
del x, y, z
if device.type == "cuda":
    torch.cuda.empty_cache()

## 3. Load a small model via Transformers

Using `EleutherAI/pythia-70m` — small enough to load fast, big enough to be a real transformer. First run downloads ~150 MB of weights.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "EleutherAI/pythia-70m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME}: {n_params / 1e6:.1f}M params on {device}")

## 4. Forward pass — English + Arabic prompts

Pythia is English-trained, so Arabic outputs will be junk. The point here is *the forward pass runs without error* and the tokenizer handles Arabic input — not that the predictions are good.

In [ ]:
prompts = {
    "english":          "The capital of Egypt is",
    "msa_fusha":        "عاصمة مصر هي",       # MSA: 'The capital of Egypt is'
    "masri_egyptian":   "عاصمة مصر هي إيه؟",  # Masri: 'What is the capital of Egypt?'
}

for label, prompt in prompts.items():
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    print(f"[{label}] tokens={inputs.input_ids.shape[1]:>2}  logits_shape={tuple(outputs.logits.shape)}")
    print(f"  prompt: {prompt}")
    print(f"  tokens: {tokenizer.convert_ids_to_tokens(inputs.input_ids[0])}")
    print()

## 5. (Optional) TransformerLens load — confirms the mech interp library works

TransformerLens is the canonical mech interp library this repo will lean on. This cell loads the same model through `HookedTransformer` to confirm the setup.

In [ ]:
try:
    from transformer_lens import HookedTransformer

    tl_model = HookedTransformer.from_pretrained("pythia-70m", device=device)
    print(f"TransformerLens load OK — n_layers={tl_model.cfg.n_layers}, d_model={tl_model.cfg.d_model}")
    del tl_model
    if device.type == "cuda":
        torch.cuda.empty_cache()
except ImportError:
    print("transformer-lens not installed yet. Install with: uv pip install transformer-lens")
except Exception as e:
    print(f"TransformerLens load failed: {type(e).__name__}: {e}")
    print("This is OK for now — Pythia loads fine via plain Transformers above.")

## Done

If all cells above passed, the environment is ready. Next:

- **Phase 1 starts at:** `notebooks/01-tokenizer-comparison-msa-vs-masri.ipynb` *(forthcoming)*
- See `README.md` → Roadmap for the full arc.

If something failed, the issue is almost certainly in the ROCm install. Check `~/code/strix-halo-ml-setup.sh` and the PyTorch ROCm nightly index.